# Metodologia de clusters de clientes LGF

Este notebook documenta el modulo vigente de segmentacion de clientes. El objetivo
no es agrupar toda la empresa en un solo modelo: primero se asigna cada cliente a
un mercado de negocio y luego se forman clusters **dentro de ese mercado** y
**dentro del ano seleccionado**.

El ejecutor oficial es `run_clusters.py`.

## 1. Flujo, mercados y ventana temporal

```text
bases de datos historicas/historic_sales_acum.csv (receta/estructura completa)
                 |
resultados/descriptivos/historico_confirmado.csv (tipologia clasificada desde fuente)
                 |
                 | filtrar --year ANO
                 v
Mercados definidos:
  01_ESTADOS_UNIDOS_CANADA
  02_THE_NETHERLANDS
  03_POLONIA
  04_ASIA
  05_OTROS
                 |
                 v
K-medias / K-modas / Jerarquico por mercado
                 |
                 v
resultados/clusters/<ANO>/
```

Se clusteriza un ano a la vez porque los clusters describen tipologias actuales de
cliente. Mezclar anos ocultaria cambios de portafolio, constancia o volumen. El
dashboard permite comparar resultados de distintos anos ya ejecutados.

**Dependencia critica:** clusters no reclasifica recetas desde la base cruda;
consume el descriptivo. Si cambia la tipologia o la fuente acumulada, se debe
regenerar `resultados/descriptivos` antes de volver a ejecutar cada ano.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

YEAR = 2026  # Cambia el ano despues de ejecutar run_clusters.py para ese periodo.
OUTPUT_DIR = ROOT / "resultados" / "clusters" / str(YEAR)

def read_cluster_output(name):
    path = OUTPUT_DIR / f"{name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"No existe {path}. Ejecuta run_clusters.py --year {YEAR}.")
    return pd.read_csv(path, low_memory=False)

## 2. Variables utilizadas

`src/lgf_operativo/clustering.py::build_client_clustering_dataset()` crea una fila
por cliente con bloques de comportamiento:

| Bloque | Variables representativas | Lectura comercial |
|---|---|---|
| Constancia | semanas activas, estabilidad, variacion reciente | Si la cuenta es programable o irregular |
| Volumen | tallos totales, tallos promedio semanal | Relevancia y escala de la cuenta |
| Colores | concentracion y diversidad de color | Portafolio enfocado o amplio |
| Productos | concentracion/diversidad de producto | Especializacion o amplitud |
| Tipo de pedido | mezcla de `SOLIDO`, `SURTIDO`, `SURTIDO_M`, `RAINBOW`, `BOUQUET`, `BQT`, `COMBO`, `BULK` | Forma de atender al cliente |
| SKU | concentracion de estructuras/SKUs | Repetibilidad comercial y operativa |
| Operacion | tallos por ramo, ramos por caja, complejidad | Esfuerzo de preparacion |
| Cumplimiento | cumplimiento de tallos | Riesgo de servicio |

`SURTIDO`, `SURTIDO_M`, `RAINBOW`, `BOUQUET`, `BQT` y `COMBO` permanecen
visibles como formatos y se tratan como estructuras mixtas. La referencia
tipologica usa los campos originales conservados en la base acumulada. No se
confunden con clientes de solidos.

In [ ]:
features = read_cluster_output("cluster_features_cliente")
features.groupby("mercado_cluster", as_index=False).agg(
    clientes=("cod_cliente", "nunique"),
    tallos=("tallos_total", "sum"),
    ventas_usd=("ventas_usd_total", "sum"),
)

## 3. Modelos evaluados y seleccion

Para cada mercado se comparan los tres metodos revisados en clase:

1. **K-medias:** apropiado cuando las variables escaladas generan perfiles
   numericos relativamente compactos.
2. **K-modas:** referencia para atributos discretizados/categoricos.
3. **Clustering jerarquico:** permite formar grupos por proximidad progresiva.

El modulo prueba alternativas de `k`, revisa medidas de separacion y penaliza
segmentaciones con clusters unitarios o demasiado dominantes. La eleccion se
hace dentro de cada mercado, por lo que un mercado puede seleccionar un metodo
diferente a otro.

In [ ]:
evaluation = read_cluster_output("cluster_model_evaluation")
cols = [c for c in [
    "mercado_cluster", "metodo", "k", "silhouette", "calinski_harabasz",
    "selection_score", "selected", "min_cluster_size", "max_cluster_share"
] if c in evaluation.columns]
evaluation[cols].sort_values(["mercado_cluster", "selection_score"], ascending=[True, False])

## 4. Resultado y lectura ejecutiva

El dashboard debe leerse en este orden:

1. Seleccionar ano y mercado.
2. Revisar el nombre/perfil ejecutivo del cluster.
3. Leer las variables que mas lo diferencian frente al promedio de su mercado.
4. Revisar clientes representativos y clientes similares.
5. Convertir el hallazgo en accion comercial, operativa o de planeacion.

Un cluster no significa automaticamente "bueno" o "malo". Puede representar,
por ejemplo, clientes de alto volumen y alta recurrencia, cuentas de composicion
compleja, o clientes ocasionales con oportunidad de estandarizacion.

In [ ]:
clusters = read_cluster_output("clusters_clientes")
resumen = read_cluster_output("cluster_resumen")

display(resumen.head(20))
display(clusters[[
    c for c in [
        "mercado_cluster", "cluster_id", "nombre_cluster", "cod_cliente",
        "cliente", "tallos_total", "tipo_pedido_dominante",
        "producto_dominante", "color_dominante"
    ] if c in clusters.columns
]].head(20))

## 5. Por que se parecen y quienes son similares

`cluster_variables_diferenciadoras.csv` compara el promedio del cluster contra su
mercado. La columna de importancia indica que variables ayudan mas a distinguir el
grupo; no equivale a causalidad.

`clientes_similares.csv` muestra vecinos dentro del mismo mercado. Sirve para
identificar cuentas que pueden recibir propuestas comerciales o validaciones
operativas parecidas.

In [ ]:
diferenciadoras = read_cluster_output("cluster_variables_diferenciadoras")
similares = read_cluster_output("clientes_similares")

display(diferenciadoras.head(30))
display(similares.head(20))

top_diff = diferenciadoras.groupby("cluster_id", group_keys=False).head(5)
px.bar(
    top_diff,
    x="importancia_modelo",
    y="lectura",
    color="cluster_id",
    orientation="h",
    title="Variables que mas explican cada cluster",
)

## 6. Decisiones que soporta el modulo

| Senal del cluster | Decision comercial | Decision operativa/planeacion |
|---|---|---|
| Alta constancia y concentracion en pocos SKU | Proponer programa recurrente | Preparar estructuras base |
| Alto volumen y diversidad de colores | Asegurar agenda comercial | Revisar disponibilidad por color |
| Dominancia `SURTIDO`/`BOUQUET`/`BQT`/`COMBO`/`RAINBOW` | Vender receta o estructura, no solo color | Validar componentes y armado |
| Caida reciente o alta variabilidad | Contactar y diagnosticar demanda | No sobredimensionar preparacion |
| Bajo cumplimiento | Gestionar expectativa/servicio | Investigar faltantes recurrentes |

## 7. Ejecucion oficial en Bash

```bash
# Este modulo supone que run_descriptivos.py ya termino con la tipologia fuente.
python run_clusters.py --input-dir "resultados/descriptivos" --year 2026
```

Para habilitar comparacion anual en el dashboard:

```bash
for anio in 2021 2022 2023 2024 2025 2026; do
  python run_clusters.py --input-dir "resultados/descriptivos" --year "$anio"
done
```

Si cambia la base o la clasificacion operativa, primero se deben regenerar los
descriptivos.